In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import glob
import shelve
import pickle
import numpy as np
import torch
import pandas as pd

from datetime import timedelta

from torch.nn import functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import json
from raven.model import GPTConfig, GPT
from tqdm import tqdm


from raven.dataset import UnifiedSeqEHRDataset

In [ ]:
# ── Configurable paths ──
NYU_DATA_DIR = "./data"  # Path to NYU processed data
NYU_PROCESSED_DIR = os.path.join(NYU_DATA_DIR, "processed_parts_final_v4")
EHRSHOT_DIR = "./ehrshot_data"  # Path to EHRSHOT dataset
EHRSHOT_MEDS_DIR = os.path.join(EHRSHOT_DIR, "EHRSHOT_MEDS")
EHRSHOT_ASSETS_DIR = os.path.join(EHRSHOT_DIR, "EHRSHOT_ASSETS")
MAPPING_DIR = "./mapping"  # Path to code mapping files
CAPSTONE_DIR = "./data"  # Path to processed data
MODEL_OUT_DIR = "./output"  # Path to model checkpoints
EHRSHOT_TASK = "new_pancan"  # Task used for both cohort construction and scoring


In [ ]:
final_header = os.path.join(NYU_PROCESSED_DIR, 'data_files/filtered_headers_token_merge.json')
# need this for converding indices to tokens
with open(final_header, 'rb') as file:
    final_headers = json.load(file)
header_to_ind = {key: i for i, key in enumerate(final_headers) }

ind_to_header = {i:key for i, key in enumerate(final_headers)}


In [ ]:
len(header_to_ind)

In [ ]:
label_df = pd.read_parquet(os.path.join(EHRSHOT_MEDS_DIR, f'labels/{EHRSHOT_TASK}/labels.parquet'))


In [ ]:
label_df

In [ ]:
ehrshotcsv = pd.read_csv(os.path.join(EHRSHOT_ASSETS_DIR, 'data/ehrshot.csv'))


In [ ]:
ehrshotcsv

In [ ]:
ehrshotcsv[ehrshotcsv['code'].str.startswith('Domain')]

In [ ]:
# in the code column, extract all the unique types of code and the count of each type, the type is indicated by the string before the first /
split_codes = ehrshotcsv['code'].str.split('/')
split_codes.str[0].value_counts()
# group by each type and count the number of unique subtypes in each type
ehrshotcsv['type'] = split_codes.str[0]
ehrshotcsv.groupby('type')['code'].nunique()

In [ ]:
split_codes.str[0].value_counts()

In [ ]:
# for ehrshotcsv, drop the column unnamed, end, omop_table
ehrshotcsv = ehrshotcsv.drop(columns=['Unnamed: 0', 'end', 'omop_table', 'type'], errors='ignore')

# remove the rows that have code starting with CARE_SITE, ICD10PCS
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CARE_SITE')] # care sites
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CPT4')] # procedures
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('ICD10PCS')] # procedures
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('ICD9Proc')] # procedures
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CVX')] # vaccines
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Visit')] # visit types
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('HCPCS')] # medical equipment
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Domain')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Medicare Specialty')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('RxNorm Extension')] # RxNorm Extension/OMOP2028607
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('CMS Place of Service')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Cancer Modifier')] # cancer staging
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('OMOP Extension')]
ehrshotcsv = ehrshotcsv[~ehrshotcsv['code'].str.startswith('Condition Type')]


In [ ]:
ehrshotcsv

In [ ]:
ehrshotcsv['code'].str.split('/').str[0].value_counts()

In [ ]:
# write csv to 
ehrshotcsv.to_csv('EHRSHOT_Data/ehrshot_cleaned.csv', index=False)

In [ ]:
# load cleaned csv
ehrshotcsv = pd.read_csv('EHRSHOT_Data/ehrshot_cleaned.csv')

# Translating Code Names

In [ ]:
# Initialize the new column with NA
ehrshotcsv['target_token_id'] = pd.NA

In [ ]:
ehrshotcsv

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

## Race, Ethinicity, and Gender

In [ ]:
# Define the translation map
# Add more mappings to this dictionary as needed
ehrshot_to_target_map = {
    # Race 1 American Indian does not exist in target
    "Race/2": "demographics_RACE_Asian",
    "Race/3": "demographics_RACE_African American (Black)",
    "Race/4": "demographics_RACE_Native Hawaiian or Other Pacific Islander",
    "Race/5": "demographics_RACE_White",
    # no code from ehrshot for unknown race
    "Ethnicity/Hispanic": "demographics_ETHNICITY_Spanish/Hispanic Origin",
    "Ethnicity/Not Hispanic" : "demographics_ETHNICITY_Not of Spanish/Hispanic Origin",
    "Gender/F": "demographics_GENDER_Female",
    "Gender/M": "demographics_GENDER_Male"
}

In [ ]:
# Apply the translation only for rows with codes in our map
# Find rows that need translation
mask = ehrshotcsv['code'].isin(ehrshot_to_target_map.keys())

# Get the target code string (e.g., "demographics_RACE_Asian")
target_codes_series = ehrshotcsv.loc[mask, 'code'].map(ehrshot_to_target_map)

# Map the target code string to the token ID using header_to_ind
# codes not in header_to_ind will become NaN
target_token_ids = target_codes_series.map(header_to_ind)

# Assign the token IDs to the new column
ehrshotcsv.loc[mask, 'target_token_id'] = target_token_ids

# Show the results for verification
print(f"Updated {mask.sum()} rows.")
print(ehrshotcsv.loc[mask, ['code', 'target_token_id']])

In [ ]:
ehrshotcsv

## Medications

In [ ]:
nyu_medication_mapping = pd.read_csv(os.path.join(MAPPING_DIR, "NDC_ATC_RXNORM_to_md_name_20251007.csv"))
nyu_medication_mapping.drop("Unnamed: 0", axis=1, inplace=True)


In [ ]:
nyu_medication_mapping

In [ ]:
# for resolving conflicts when multiple tokens can be mapped
from collections import Counter
import shelve
from tqdm import tqdm

# Load training data info
train_df_path = os.path.join(NYU_PROCESSED_DIR, "csvs/train.csv")
train_df = pd.read_csv(train_df_path)

db_folder = NYU_PROCESSED_DIR
# Open shelves
db_dict = {part: shelve.open(db_folder+'/processed_part_'+str(part)+'.shelve') for part in tqdm(range(40), desc="Opening shelves")}

# Load mask
remove_inds = np.load(os.path.join(NYU_PROCESSED_DIR, "data_files/delete_inds.npy"))
total_prev_tokens = 57735
keep_mask = np.ones(total_prev_tokens, dtype=bool)
keep_mask[remove_inds] = False

# Count token frequencies
print("Counting token frequencies in training data...")
token_counts = Counter()

# Iterate by part for efficiency
for part_id in range(40):
    # Filter patients in this part
    part_patients = train_df[train_df['part'] == part_id]['patient_id']
    
    if len(part_patients) == 0:
        continue
        
    shelve_file = db_dict[part_id]
    
    for pid in tqdm(part_patients, desc=f"Part {part_id}", leave=False):
        try:
            # Assuming patient_id in CSV matches key in shelve
            p_key = str(pid) 
            if p_key in shelve_file:
                a = shelve_file[p_key]
                # Apply mask (columns)
                a_reduced = a[:, keep_mask]
                
                # Get non-zero indices (tokens)
                if hasattr(a_reduced, "nonzero"):
                     rows, token_arr = a_reduced.nonzero()
                else:
                     rows, token_arr = np.nonzero(a_reduced)

                # Convert to simple list/array and update
                token_counts.update(token_arr)
        except Exception as e:
            continue

print(f"Finished counting. Found {len(token_counts)} unique tokens.")


In [ ]:
# save token counts dict
with open('EHRSHOT_Data/token_counts.pkl', 'wb') as f:
    import pickle
    pickle.dump(token_counts, f)


In [ ]:
# 1. Build med_key_to_token_id from header_to_ind
# Valid keys start with "Medication:" and end with a number
med_key_to_token_id = {}
for header, idx in header_to_ind.items():
    if str(header).startswith("Medication:"):
        try:
            # Extract the key at the end (e.g., "Medication:NAME 123" -> 123)
            key_str = str(header).rsplit(' ', 1)[-1]
            key_int = int(key_str)
            med_key_to_token_id[key_int] = idx
        except ValueError:
            # Some headers might not match the format perfectly, skip them
            continue

print(f"Found {len(med_key_to_token_id)} Medication keys in header_to_ind")

# 2. Build rxnorm_to_med_key with conflict resolution
print("Building optimized RxNorm mapping...")

# Filter for RxNorm
rxnorm_map_df = nyu_medication_mapping[nyu_medication_mapping['type'] == 'RxNorm'].copy()
rxnorm_map_df['code_str'] = rxnorm_map_df['code'].astype(str)

# We want to map code_str -> single med_key
# But first, filter to med_keys that exist in med_key_to_token_id
valid_med_keys = set(med_key_to_token_id.keys())

# Group by RxNorm code to see all possible VALID medication keys
# Create a mapping: RxNorm Code -> List of Valid Medication Keys
code_to_valid_keys = {}

# Optimized grouping
# Filter rows first
valid_rows = rxnorm_map_df[rxnorm_map_df['medicationkey'].isin(valid_med_keys)]

for code, group in valid_rows.groupby('code_str'):
    keys = group['medicationkey'].unique()
    code_to_valid_keys[code] = keys

# Resolve conflicts using token frequency
rxnorm_to_med_key = {}
resolved_counts = 0
total_conflicts = 0

for code, keys in code_to_valid_keys.items():
    if len(keys) == 1:
        rxnorm_to_med_key[code] = keys[0]
    else:
        # Multiple valid tokens for this RxNorm code. Pick the most frequent one.
        total_conflicts += 1
        best_key = None
        max_count = -1
        
        for k in keys:
            token_id = med_key_to_token_id[k]
            # Use computed token_counts
            count = token_counts.get(token_id, 0)
            if count > max_count or (count == max_count and best_key is None):
                max_count = count
                best_key = k
        
        rxnorm_to_med_key[code] = best_key
        resolved_counts += 1

print(f"Mapped {len(rxnorm_to_med_key)} RxNorm codes to unique medication keys.")
print(f"Resolved {total_conflicts} conflicts using frequency data.")

# 3. Create mapping from full ehrshot code string to token_id
# We only care about codes starting with "RxNorm/"
rxnorm_codes_ehrshot = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('RxNorm/'), 'code'].unique()

count_mapped = 0
rxnorm_translation_map = {}

for full_code in rxnorm_codes_ehrshot:
    # Extract code part: "RxNorm/123" -> "123"
    code_val = full_code.split('/')[1]
    
    if code_val in rxnorm_to_med_key:
        med_key = rxnorm_to_med_key[code_val]
        if med_key in med_key_to_token_id:
            token_id = med_key_to_token_id[med_key]
            rxnorm_translation_map[full_code] = token_id
            count_mapped += 1

print(f"Mapped {count_mapped} out of {len(rxnorm_codes_ehrshot)} unique RxNorm codes found in EHRSHOT to target tokens.")

# 4. Apply the translation to the dataframe
# Re-use the 'target_token_id' column we created earlier, filling in NaNs where applicable

# Find rows with RxNorm codes that we can translate
mask_rx = ehrshotcsv['code'].isin(rxnorm_translation_map.keys())

# Map directly to token_id (since our map values are already token_ids)
ehrshotcsv.loc[mask_rx, 'target_token_id'] = ehrshotcsv.loc[mask_rx, 'code'].map(rxnorm_translation_map)

print(f"Updated {mask_rx.sum()} rows with Medication translations.")
print(ehrshotcsv.loc[mask_rx, ['code', 'target_token_id']].head())

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

In [ ]:
import pandas as pd

# 1. Keep only RxNorm rows from your events table
rx_events = ehrshotcsv[ehrshotcsv["code"].str.startswith("RxNorm/")].copy()

# Extract the numeric/string RxNorm code part after "RxNorm/"
rx_events["rxnorm_code"] = rx_events["code"].str.split("/").str[1]

# 2. Keep only RxNorm rows from your mapping table
rx_map = nyu_medication_mapping[nyu_medication_mapping["type"] == "RxNorm"].copy()

# Make sure both sides are strings so merge works even if some codes look like 10370-171-09 or C08C
rx_events["rxnorm_code"] = rx_events["rxnorm_code"].astype(str)
rx_map["code"] = rx_map["code"].astype(str)

# 3. Merge to see which events can be translated
rx_events_merged = rx_events.merge(
    rx_map[["code", "medicationkey", "name"]],
    left_on="rxnorm_code",
    right_on="code",
    how="left",
    suffixes=("", "_map"),
)

# 4. Coverage at the *event* level
n_events = len(rx_events_merged)
n_mapped_events = rx_events_merged["medicationkey"].notna().sum()
event_coverage = n_mapped_events / n_events if n_events > 0 else float("nan")

print(f"RxNorm events:        {n_events:,}")
print(f"Mapped events:        {n_mapped_events:,} ({event_coverage:.2%})")

# 5. Coverage at the *unique code* level
used_codes = rx_events_merged["rxnorm_code"].unique()
n_codes = len(used_codes)

mapped_codes = rx_map[rx_map["code"].isin(used_codes)]["code"].unique()
n_mapped_codes = len(mapped_codes)
code_coverage = n_mapped_codes / n_codes if n_codes > 0 else float("nan")

print(f"Unique RxNorm codes:  {n_codes:,}")
print(f"Mapped unique codes:  {n_mapped_codes:,} ({code_coverage:.2%})")

# 6. (Optional) peek at some unmapped codes
unmapped_codes = sorted(set(used_codes) - set(mapped_codes))
print("Example unmapped codes:", unmapped_codes[:20])

In [ ]:
# Analyze top 20 most frequent RxNorm codes
top_n = 20
rx_counts = ehrshotcsv.loc[ehrshotcsv['code'].str.startswith('RxNorm/'), 'code'].value_counts().head(top_n)

# Prepare a list to store details
results = []

for code_full, count in rx_counts.items():
    # Check mapping status
    is_mapped = code_full in rxnorm_translation_map
    
    # Attempt to retrieve the medication name
    # code_full format is "RxNorm/{code}"
    code_val = code_full.split('/')[1]
    
    # Look up name in the mapping dataframe (loaded in previous cells)
    name_matches = nyu_medication_mapping.loc[
        (nyu_medication_mapping['type'] == 'RxNorm') & 
        (nyu_medication_mapping['code'].astype(str) == code_val),
        'name'
    ]
    
    name = name_matches.iloc[0] if not name_matches.empty else "Unknown"
    
    results.append({
        "EHRSHOT Code": code_full,
        "Name": name,
        "Frequency": count,
        "Mapped": is_mapped
    })

# Display results
top_rx_df = pd.DataFrame(results)
print(f"Top {top_n} Most Frequent RxNorm Codes in EHRSHOT data:")
display(top_rx_df)

In [ ]:
rxnorm_to_med_key['313782']

In [ ]:
nyu_medication_mapping[nyu_medication_mapping['code'] == '313782']

## Diagnosis

### ICD-O-3

In [ ]:
# 1. Build diag_code_to_token_id from header_to_ind
diag_code_to_token_id = {}
duplicate_diag_codes = 0

for header, idx in header_to_ind.items():
    if str(header).startswith("Diagnosis:"):
        # Assumption: The code is the last part of the string separated by space
        code_part = str(header).split(' ')[-1]
        
        # Determine if we should handle duplicates
        # Ideally, codes should be unique or we loop through and overwrite/ignore.
        # Let's map code -> token_id
        if code_part in diag_code_to_token_id:
            duplicate_diag_codes += 1
            pass 
        
        diag_code_to_token_id[code_part] = idx

print(f"Found {len(diag_code_to_token_id)} Diagnosis codes in header_to_ind (Duplicates skipped/overwritten: {duplicate_diag_codes})")

# 2. Build icdo3_translation_map
icdo3_codes_ehrshot = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('ICDO3/'), 'code'].unique()

print(f"Found {len(icdo3_codes_ehrshot)} unique ICDO3-related codes in EHRSHOT")

icdo3_translation_map = {}
mapped_count_icd = 0

for full_code in icdo3_codes_ehrshot:
    full_code_str = str(full_code)
    
    # Logic to extract code:
    # If '-' exists, take part after last '-'
    # Else take part after last '/'
    if '-' in full_code_str:
        extracted_code = full_code_str.split('-')[-1]
    else:
        extracted_code = full_code_str.split('/')[-1]
        
    if extracted_code in diag_code_to_token_id:
        token_id = diag_code_to_token_id[extracted_code]
        icdo3_translation_map[full_code] = token_id
        mapped_count_icd += 1

print(f"Mapped {mapped_count_icd} out of {len(icdo3_codes_ehrshot)} unique ICDO3 codes to target tokens.")

# 3. Apply the translation
mask_icd = ehrshotcsv['code'].isin(icdo3_translation_map.keys())
ehrshotcsv.loc[mask_icd, 'target_token_id'] = ehrshotcsv.loc[mask_icd, 'code'].map(icdo3_translation_map)

print(f"Updated {mask_icd.sum()} rows with ICDO3 translations.")
print(ehrshotcsv.loc[mask_icd, ['code', 'target_token_id']].head())

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

### SNOMED

In [ ]:
# read in tsv file
snomed_icd_mapping = pd.read_csv(os.path.join(MAPPING_DIR, "tls_Icd10cmHumanReadableMap_US1000124_20250901.tsv"), sep="\t")

In [ ]:
# 1. Prepare snomed_icd_mapping lookup
# Filter for active mappings and mapGroup 1
active_map = snomed_icd_mapping[(snomed_icd_mapping['active'] == 1) & (snomed_icd_mapping['mapGroup'] == 1)].copy()
active_map = active_map.sort_values(by=['referencedComponentId', 'mapPriority'])
active_map = active_map.drop_duplicates(subset=['referencedComponentId'], keep='last')

# Create dict: SNOMED Code (int) -> ICD 10 Code (str)
snomed_int_to_icd = dict(zip(active_map['referencedComponentId'], active_map['mapTarget']))

print(f"Created mapping dictionary with {len(snomed_int_to_icd)} SNOMED entries (using mapGroup=1, last mapPriority).")

# 2. Build SNOMED -> Token ID Map
snomed_translation_map = {}
mapped_count_snomed = 0

# diagnosis codes already in diag_code_to_token_id from previous step
# diag_code_to_token_id keys are ICD10 codes (e.g. "C56.9")

snomed_codes_in_ehr_unique = ehrshotcsv.loc[ehrshotcsv['code'].astype(str).str.startswith('SNOMED/'), 'code'].unique()

for full_code in snomed_codes_in_ehr_unique:
    # Full code: e.g., "SNOMED/223003"
    snomed_str = full_code.split('/')[-1]
    
    try:
        snomed_int = int(snomed_str)
    except ValueError:
        continue
        
    # Get mapped ICD
    if snomed_int in snomed_int_to_icd:
        icd_target = snomed_int_to_icd[snomed_int]
        # mapTarget might be non-string or NaN, ensure string
        if pd.isna(icd_target):
            continue
        icd_target = str(icd_target)
        
        # Try finding in our header mapping
        if icd_target in diag_code_to_token_id:
            token_id = diag_code_to_token_id[icd_target]
            snomed_translation_map[full_code] = token_id
            mapped_count_snomed += 1

print(f"Mapped {mapped_count_snomed} out of {len(snomed_codes_in_ehr_unique)} unique SNOMED codes to target tokens.")

# 3. Apply translations (Fill only where currently NaN to avoid overwriting previous work, 
# though SNOMED rows likely stem from different rows than Race/Rx/ICDO3)
mask_snomed = ehrshotcsv['code'].isin(snomed_translation_map.keys())

# We can overwrite or fillna. Since code column is unique per row, overwrite is fine.
ehrshotcsv.loc[mask_snomed, 'target_token_id'] = ehrshotcsv.loc[mask_snomed, 'code'].map(snomed_translation_map)

print(f"Updated {mask_snomed.sum()} rows with SNOMED translations.")
print(ehrshotcsv.loc[mask_snomed, ['code', 'target_token_id']].head())

In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

## Labs

Fit lab normalization using only subjects in the official EHRSHOT training split.
The MEDS split file is checked against the original EHRSHOT split CSV. The same
training mean and sample standard deviation are then used for every split.
Only finite numeric measurements contribute to the statistics; observed zeros
remain valid. Missing measurements, labs without positive training variance,
unavailable vocabulary bins, and normalized values outside `[-10, 10)` remain
unmapped. No NYU lab statistics or unit conversion are used.


In [ ]:
"""Normalize EHRSHOT labs using only the official training subjects.

Statistics use finite measurements, including observed zeros, and sample
standard deviation (ddof=1). There is no NYU fallback or unit conversion.
"""

import re

import numpy as np
import pandas as pd


LAB_BINS = np.array([-10, -3, -1, -0.5, 0.5, 1, 3, 10])


def load_ehrshot_splits(meds_path, original_path):
    """Validate the MEDS splits against EHRSHOT's original subject split file."""
    meds = pd.read_parquet(meds_path)[['subject_id', 'split']]
    meds['split'] = meds['split'].replace({'tuning': 'val', 'held_out': 'test'})
    if (meds.isna().any().any() or meds.subject_id.duplicated().any()
            or set(meds.split) != {'train', 'val', 'test'}):
        raise ValueError('Invalid or overlapping MEDS splits')
    original = pd.read_csv(original_path).rename(columns={'omop_person_id': 'subject_id'})
    original = original[['subject_id', 'split']]
    splits = dict(zip(meds.subject_id, meds.split))
    if (original.isna().any().any() or original.subject_id.duplicated().any()
            or dict(zip(original.subject_id, original.split)) != splits):
        raise ValueError('MEDS and original EHRSHOT splits disagree')
    return splits


def lab_vocabulary(header_to_ind):
    """Map each LOINC to its feature name and seven target token IDs.

    Public vocabularies can omit bins; those bins remain unmapped (-1).
    """
    labs = {}
    intervals = list(zip(LAB_BINS[:-1], LAB_BINS[1:]))
    for header, token_id in header_to_ind.items():
        match = re.fullmatch(r'(Lab:.+)\[normedValue_sd_between_([^,]+),([^\]]+)\]', str(header))
        if not match:
            continue
        feature, low, high = match.groups()
        code = 'LOINC/' + feature.rsplit(' ', 1)[-1]
        bin_index = intervals.index((float(low), float(high)))
        item = labs.setdefault(code, {'feature': feature, 'tokens': [-1] * 7})
        if item['feature'] != feature or item['tokens'][bin_index] != -1:
            raise ValueError(f'Ambiguous lab vocabulary: {code}')
        item['tokens'][bin_index] = int(token_id)
    return labs


def fit_training_lab_stats(lab_events, splits):
    """Fit one distribution per code from train subjects; exclude unsplit IDs.

    ``lab_events`` must contain patient_id, code, and value. The returned stats
    can be reused to translate every split without fitting again.
    """
    train_ids = {patient for patient, split in splits.items() if split == 'train'}
    if not train_ids:
        raise ValueError('No EHRSHOT training subjects in the supplied splits')
    train = lab_events.loc[lab_events.patient_id.isin(train_ids), ['code', 'value']].copy()
    train['value'] = pd.to_numeric(train.value, errors='coerce').astype(float)
    train = train.loc[np.isfinite(train.value)]
    grouped = train.groupby('code').value.agg(['count', 'mean', 'std'])
    grouped = grouped.reindex(lab_events.code.unique())
    result = {}
    for code, row in grouped.iterrows():
        count = int(row['count']) if pd.notna(row['count']) else 0
        mean = float(row['mean']) if count else None
        std = float(row['std']) if count > 1 else None
        result[code] = {
            'count': count,
            'mean': mean,
            'std': std,
            'usable': bool(count > 1 and np.isfinite(mean)
                           and np.isfinite(std) and std > 0),
        }
    return result


def bin_lab_values(values, mean, std, tokens):
    """Return token IDs for frozen statistics, or -1 for unmappable values."""
    values = np.asarray(values, dtype=float)
    result = np.full(len(values), -1, dtype=np.int64)
    if (mean is None or std is None or not np.isfinite(mean)
            or not np.isfinite(std) or std <= 0):
        return result
    with np.errstate(invalid='ignore', over='ignore'):
        normalized = (values - mean) / std
    bins = np.searchsorted(LAB_BINS, normalized, side='right') - 1
    valid = np.isfinite(normalized) & (bins >= 0) & (bins < 7)
    result[valid] = np.asarray(tokens, dtype=np.int64)[bins[valid]]
    return result


In [ ]:
ehrshot_splits = load_ehrshot_splits(
    os.path.join(EHRSHOT_MEDS_DIR, 'metadata', 'subject_splits.parquet'),
    os.path.join(EHRSHOT_ASSETS_DIR, 'splits', 'person_id_map.csv'),
)
lab_vocab = lab_vocabulary(header_to_ind)
ehrshot_labs = ehrshotcsv.loc[
    ehrshotcsv['code'].isin(lab_vocab), ['patient_id', 'code', 'value']
].copy()
lab_stats_train = fit_training_lab_stats(ehrshot_labs, ehrshot_splits)

# Apply the frozen training statistics to train, validation, and test events.
# Clear earlier lab assignments so rerunning this cell cannot retain stale tokens.
lab_mask = ehrshotcsv['code'].astype(str).str.startswith('LOINC/')
ehrshotcsv.loc[lab_mask, 'target_token_id'] = pd.NA
lab_token_ids = pd.Series(pd.NA, index=ehrshot_labs.index, dtype='Int64')
for code, group in ehrshot_labs.groupby('code', sort=False):
    stats = lab_stats_train[code]
    values = pd.to_numeric(group['value'], errors='coerce').astype(float).to_numpy()
    tokens = bin_lab_values(values, stats['mean'], stats['std'], lab_vocab[code]['tokens'])
    valid = tokens >= 0
    lab_token_ids.loc[group.index[valid]] = tokens[valid]

lab_token_ids = lab_token_ids.dropna()
ehrshotcsv.loc[lab_token_ids.index, 'target_token_id'] = lab_token_ids
usable_count = sum(stats['usable'] for stats in lab_stats_train.values())
print(f'Usable training statistics for {usable_count} of {len(lab_vocab)} lab codes.')
print(f'Mapped {len(lab_token_ids)} of {len(ehrshot_labs)} supported lab rows.')
display(ehrshotcsv.loc[lab_token_ids.index[:5], ['code', 'value', 'target_token_id']])


In [ ]:
# Coverage diagnostics use the same training-only statistics as translation.
top_n = 20
loinc_counts = ehrshotcsv.loc[lab_mask, 'code'].value_counts().head(top_n)
results_loinc = []
for code, count in loinc_counts.items():
    stats = lab_stats_train.get(code, {})
    results_loinc.append({
        'EHRSHOT Code': code,
        'Feature Name': lab_vocab.get(code, {}).get('feature', 'Unknown / Not in Header'),
        'Frequency': count,
        'In Header': code in lab_vocab,
        'Training Measurements': stats.get('count', 0),
        'Training Mean': stats.get('mean'),
        'Training Std': stats.get('std'),
        'Usable Training Stats': stats.get('usable', False),
    })
top_loinc_df = pd.DataFrame(results_loinc)
display(top_loinc_df)


In [ ]:
ehrshotcsv['target_token_id'].isna().sum()

In [ ]:
# Inspect the training statistics used to normalize body temperature.
lab_stats_train.get('LOINC/8310-5')


## End

In [ ]:
# 1. Filter out rows with NA for target_token_id
ehrshot_clean = ehrshotcsv.dropna(subset=['target_token_id'])

# Save to EHRSHOT_Data/ehrshot_mapped.csv
output_dir = 'EHRSHOT_Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

mapped_path = os.path.join(output_dir, 'ehrshot_mapped.csv')
ehrshot_clean.to_csv(mapped_path, index=False)
print(f"Saved cleaned mapped data to {mapped_path}. Row count: {len(ehrshot_clean)}")

# 2. Create subset with specific columns
subset_cols = ['patient_id', 'visit_id', 'start', 'target_token_id']
ehrshot_final = ehrshot_clean[subset_cols]

# Save to EHRSHOT_Data/ehrshot_token_ids.csv
tokens_path = os.path.join(output_dir, 'ehrshot_token_ids.csv')
ehrshot_final.to_csv(tokens_path, index=False)
print(f"Saved final tokens data to {tokens_path}. Row count: {len(ehrshot_final)}")
print("First 5 rows of final tokens:\n", ehrshot_final.head())

In [ ]:
# load ehrshot_final
output_dir = 'EHRSHOT_Data'
tokens_path = os.path.join(output_dir, 'ehrshot_token_ids.csv')
ehrshot_final = pd.read_csv(tokens_path)

# Building Eval Dataset

In [ ]:
# 1. Identify Demographics Tokens
demo_target_strings = set(ehrshot_to_target_map.values())
demo_token_ids = set()
for t_str in demo_target_strings:
    if t_str in header_to_ind:
        demo_token_ids.add(header_to_ind[t_str])

print(f"Identified {len(demo_token_ids)} demographics token IDs.")

In [ ]:
# 2. Prepare Data
# Ensure datetime
ehrshot_final['start'] = pd.to_datetime(ehrshot_final['start'])

# Split into Demo and Clinical
# We'll assume demographics tokens are EXACTLY those in our set.
# Note: ehrshot_final['target_token_id'] might need to be int.
ehrshot_final['target_token_id'] = pd.to_numeric(ehrshot_final['target_token_id'], errors='coerce')
ehrshot_final = ehrshot_final.dropna(subset=['target_token_id'])
ehrshot_final['target_token_id'] = ehrshot_final['target_token_id'].astype(int)

is_demo = ehrshot_final['target_token_id'].isin(demo_token_ids)
demo_df = ehrshot_final[is_demo].copy()
clinical_df = ehrshot_final[~is_demo].copy()
# drop token_id = 26836
age_df = ehrshot_final[ehrshot_final['target_token_id'] == 26836]
clinical_df = clinical_df[clinical_df['target_token_id'] != 26836]

print(f"Demographics rows: {len(demo_df)}")
print(f"Age rows: {len(age_df)}")
print(f"Clinical rows: {len(clinical_df)}")

# 3. Group and Process
# Pre-compute demographics for each patient: patient_id -> list of tokens
demo_grp = demo_df.groupby('patient_id')['target_token_id'].apply(list)
patient_demo_tokens = demo_grp.to_dict()

In [ ]:
# RE-PROCESS: Merge visits on the same day & Remove duplicates
from collections import defaultdict

print("Refining visits to day-level...")

# 1. Day-level grouping
# clinical_df comes from previous step
clinical_df['start_date'] = clinical_df['start'].dt.floor('D')

# Group by patient and day
# We aggregate tokens into a list, then deduplicate
print("Grouping by Patient and Date...")
day_groups = clinical_df.groupby(['patient_id', 'start_date'])['target_token_id'].apply(list)

# Deduplicate tokens within each day-visit
# We use set to remove duplicates, then convert back to list
# Sorting ensures deterministic order which is often good
print("Deduplicating tokens...")
day_groups = day_groups.apply(lambda x: sorted(list(set(x))))

# 2. Build Timelines
# Convert series to similar structure as before: dict of list of dicts
# patient_id -> [{'time': date, 'tokens': [...]}, ...]
print("Building timelines...")

patient_timelines = defaultdict(list)

# Iterate over the series
# The index is (patient_id, start_date)
# We can reset index to iterate easier or iterate items
day_groups_df = day_groups.reset_index()
day_groups_df = day_groups_df.sort_values(['patient_id', 'start_date'])

for row in day_groups_df.itertuples(index=False):
    # row has patient_id, start_date, target_token_id
    patient_timelines[row.patient_id].append({
        'time': row.start_date,
        'tokens': row.target_token_id
    })

print(f"Built timelines for {len(patient_timelines)} patients.")

In [ ]:
def get_age_bucket_header(age_years):
    if 1 <= age_years < 5: return "demographics_age[between_1_and_5]"
    if 5 <= age_years < 10: return "demographics_age[between_5_and_10]"
    if 10 <= age_years < 20: return "demographics_age[between_10_and_20]"
    if 20 <= age_years < 30: return "demographics_age[between_20_and_30]"
    if 30 <= age_years < 40: return "demographics_age[between_30_and_40]"
    if 40 <= age_years < 50: return "demographics_age[between_40_and_50]"
    if 50 <= age_years < 60: return "demographics_age[between_50_and_60]"
    if 60 <= age_years < 65: return "demographics_age[between_60_and_65]"
    if 65 <= age_years < 70: return "demographics_age[between_65_and_70]"
    if 70 <= age_years < 75: return "demographics_age[between_70_and_75]"
    if 75 <= age_years < 80: return "demographics_age[between_75_and_80]"
    if 80 <= age_years < 85: return "demographics_age[between_80_and_85]"
    if 85 <= age_years < 90: return "demographics_age[between_85_and_90]"
    if 90 <= age_years < 95: return "demographics_age[between_90_and_95]"
    if 95 <= age_years < 100: return "demographics_age[between_95_and_100]"
    if 100 <= age_years < 150: return "demographics_age[between_100_and_150]"
    return None

In [ ]:
# 4. Create Examples for Label DF
print("Creating inference examples...")

# Pre-load label_df if not already (it is in memory as label_df)
# Ensure timestamps
disease = EHRSHOT_TASK
label_df = pd.read_parquet(os.path.join(EHRSHOT_MEDS_DIR, f'labels/{disease}/labels.parquet'))
label_df['prediction_time'] = pd.to_datetime(label_df['prediction_time'])
if (label_df[['subject_id', 'prediction_time', 'boolean_value']].isna().any().any()
        or label_df.duplicated(['subject_id', 'prediction_time']).any()):
    raise ValueError('Missing or duplicate EHRSHOT labels')
if not set(label_df['subject_id']) <= set(ehrshot_splits):
    raise ValueError('Labeled subjects missing from the official EHRSHOT splits')


# Prepare DOB map
# Ensure 'start' is datetime
age_df['start'] = pd.to_datetime(age_df['start'], errors='coerce')
dobs = age_df.set_index('patient_id')['start'].to_dict()

inference_data = []

count_no_history = 0

# Iterate labels
for _, row in label_df.iterrows():
    pid = row['subject_id']
    pred_time = row['prediction_time']
    label = row['boolean_value']
    
    if pid not in patient_timelines:
        count_no_history += 1
        continue
        
    timeline = patient_timelines[pid]
    
    # Filter visits <= pred_time
    valid_visits = [v for v in timeline if v['time'] <= pred_time]
    
    if not valid_visits:
        count_no_history += 1
        continue
    
    # Get DOB
    dob = dobs.get(pid, pd.NaT)
    
    # Process valid visits to add age tokens (to ALL visits) and demographics (to FIRST visit)
    processed_valid_visits = []
    
    for i, v in enumerate(valid_visits):
        # Create a shallow copy of the visit dict so we don't modify the original timeline
        current_visit = v.copy()
        
        # Get tokens list (ensure it's a new list)
        current_tokens = list(current_visit['tokens'])
        
        # 1. Add Age Token (Dynamic, for each visit)
        if not pd.isna(dob):
            # Calculate age at time of visit
            age_years = (current_visit['time'] - dob).days / 365.25
            # print(age_years)
            
            bucket_header = get_age_bucket_header(age_years)
            if bucket_header and bucket_header in header_to_ind:
                age_token_id = header_to_ind[bucket_header]
                # Prepend age token
                # print(age_token_id)
                current_tokens.insert(0, age_token_id)
        
        # 2. Add Static Demographics (Static, only for first visit)
        if i == 0 and pid in patient_demo_tokens:
            demo_tokens = patient_demo_tokens[pid]
            # Prepend demo tokens. Note: if age was added, it is at index 0.
            # We add demo tokens before everything (result: [Demo Tokens, Age Token, Original Tokens])
            current_tokens = demo_tokens + current_tokens
            
        current_visit['tokens'] = current_tokens
        processed_valid_visits.append(current_visit)
        
    # Use our processed list
    valid_visits = processed_valid_visits

    # Calculate relative days
    t0 = valid_visits[0]['time']
    
    processed_visits = []
    
    for v in valid_visits:
        # relative days (float or int)
        # using days component of timedelta
        diff = v['time'] - t0
        days = diff.days + diff.seconds / (24*3600) 
        
        processed_visits.append({
            'days': days,
            'tokens': v['tokens'] # list of ints
        })
        
    # Create entry
    entry = {
        'subject_id': pid,
        'split': ehrshot_splits[pid],
        'prediction_time': pred_time,
        'boolean_value': label,
        'visits': processed_visits
    }
    
    inference_data.append(entry)

print(f"Generated {len(inference_data)} examples. (Skipped {count_no_history} due to no history)")

output_disease_dir = os.path.join(output_dir, disease)
os.makedirs(output_disease_dir, exist_ok=True)
for split in ('all', 'train', 'val', 'test'):
    records = inference_data if split == 'all' else [r for r in inference_data if r['split'] == split]
    suffix = '' if split == 'all' else '_' + split
    output_path = os.path.join(output_disease_dir, f'ehrshot_inference_data{suffix}.pkl')
    with open(output_path, 'wb') as f:
        pickle.dump(records, f)
    print(f'Saved {len(records)} {split} examples to {output_path}')


In [ ]:
import torch
from torch.utils.data import DataLoader
# Ensure condition_helper_ehrshot is importable
import sys
import os
sys.path.append(os.getcwd())
from data_processing.condition_helper_ehrshot import SeqCLSDatasetEHRSHOT, aggregate_logits, GPTCONFIG_PARAMS
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
import numpy as np
from tqdm import tqdm

# 1. Define Evaluation Function (Single Process)
@torch.no_grad()
def evaluate_model(model, dataloader, device, condition_indices, condition_name="Pancreatic Cancer", 
                   agg_method="sum", threshold=0.5):
    """
    Evaluate the model on a dataset.
    """
    model.eval()
    all_logits = []
    all_labels = []
    
    # Get condition indices
    cond_ind = condition_indices
    if isinstance(condition_indices, dict):
        cond_ind = condition_indices[condition_name]
        
    vocab_size = model.config.vocab_size
    if not cond_ind or any(i < 0 or i >= vocab_size for i in cond_ind):
        raise ValueError('Condition indices must be nonempty and match the checkpoint vocabulary.')
    condition_mask = torch.zeros(vocab_size, dtype=torch.bool, device=device)
    condition_mask[cond_ind] = True

    print(f"Evaluating on {len(dataloader.dataset)} examples...")
    
    for batch in tqdm(dataloader):
        # Move to device
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        # Forward pass using positional arguments as per eval_condition_mp.py
        # Args: idx, days_embed,, targets, pred_mask, pad_mask, attention_bias
        logits, _, _, _ = model(
            batch['token_arr'],
            batch['days_embed_arr'],
            None,
            batch['pred_mask'],
            batch['pad_mask'],
            batch['attention_bias'],
            patient_ids=batch.get('patient_ids', None)
        )
        
        # Get masked logits (predictions at the query position)
        masked_logits = logits[batch['pred_mask']]
        
        # Aggregate logits for the target condition specific tokens
        agg_logits = aggregate_logits(masked_logits, condition_mask, method=agg_method, dim=1)
        
        all_logits.append(agg_logits.cpu())
        all_labels.append(batch['labels'].cpu())
        
    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy().reshape(-1)
    
    # Compute Metrics
    # predictions = (all_logits >= threshold).astype(int)
    
    metrics = {
        'auroc': roc_auc_score(all_labels, all_logits),
        'auprc': average_precision_score(all_labels, all_logits)
    }
    
    return all_labels, all_logits, metrics


In [ ]:
from glob import glob
def load_latest_checkpoint(out_dir, device):
    """
    Load the latest checkpoint from the output directory.
    
    Args:
        out_dir: Directory containing checkpoints
        device: Device to load checkpoint to
        
    Returns:
        checkpoint: Loaded checkpoint
        checkpoint_model_args: Model arguments from checkpoint
        config: Configuration dictionary
    """
    checkpoint_files = glob(os.path.join(out_dir, 'ckpt-*.pt'))
    
    with open(os.path.join(out_dir, 'config.json'), 'r') as f:
        config = json.load(f)
    
    if not checkpoint_files:
        raise FileNotFoundError(f"No checkpoint files found in {out_dir}")
    
    checkpoint_files.sort(key=lambda x: int(x.split('-')[-1].split('.')[0]))
    
    latest_checkpoint = checkpoint_files[-1]
    
    print(f"Loading checkpoint: {latest_checkpoint}")
    
    checkpoint = torch.load(latest_checkpoint, map_location=device)
    # Prefer configuration saved with this checkpoint over the run directory.
    config.update(checkpoint.get('config', {}))
    checkpoint_model_args = checkpoint['model_args']
    
    return checkpoint, checkpoint_model_args, config

In [ ]:
# MODEL_OUT_DIR must point to the directory containing config.json and ckpt-*.pt.
out_dir = MODEL_OUT_DIR
condition_indices_path = os.path.join(NYU_PROCESSED_DIR, 'conditions2inds_ehrshot_plus_one.json')
with open(condition_indices_path, "r") as f:
    condition_indices = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Load model checkpoint
checkpoint, checkpoint_model_args, config = load_latest_checkpoint(out_dir, device)
print("Latest checkpoint loaded successfully")

# Create model
model_args = {}
for key in GPTCONFIG_PARAMS:
    if key in checkpoint_model_args:
        model_args[key] = checkpoint_model_args[key]

# Use the configured local metadata path when relocating a checkpoint.
model_args['data_dir'] = NYU_PROCESSED_DIR
gptconf = GPTConfig(**model_args)
model = GPT(gptconf)

# Clean up state dict if needed (remove DDP-specific prefixes)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.to(device)

# Wrap model in DDP
model.eval()


In [ ]:
# Use the checkpoint's training setting for patient-position embeddings.
multiwin = config.get('multiwin')
if not isinstance(multiwin, bool):
    raise ValueError("Checkpoint config must specify multiwin as true or false.")

# Use the same disease whose examples were generated above.
data_path = os.path.join(output_dir, disease, 'ehrshot_inference_data_test.pkl')
dataset = SeqCLSDatasetEHRSHOT(
    data_path, max_length=model.config.block_size, approach='direct',
    time_horizon=365, multiwin=multiwin,
)
if not dataset.data or any(example.get('split') != 'test' for example in dataset.data):
    raise ValueError('Regenerate the test cohort with the updated construction cells above.')
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)


In [ ]:
condition_name = {
    'new_acutemi': 'Heart Attack',
    'new_pancan': 'Pancreatic Cancer',
    'new_hypertension': 'Hypertension',
    'new_hyperlipidemia': 'Hyperlipidemia',
    'new_lupus': 'Lupus',
    'new_celiac': 'Celiac',
}[EHRSHOT_TASK]
all_labels, all_logits, metrics = evaluate_model(
    model, dataloader, device, condition_indices, condition_name=condition_name,
    agg_method='sum', threshold=0.5,
)


In [ ]:
metrics

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)

# def sigmoid(x):
#     x = np.asarray(x)
#     return 1 / (1 + np.exp(-x))

def plot_roc_pr(all_labels, all_logits, title_prefix=""):
    y_true = np.asarray(all_labels).astype(int).ravel()
    logits = np.asarray(all_logits).ravel()
    # y_score = sigmoid(logits)  # convert logits -> probabilities
    y_score = logits

    # --- ROC ---
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auroc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUROC = {auroc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix}ROC Curve".strip())
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- PR ---
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)

    # baseline = prevalence
    baseline = y_true.mean()

    plt.figure()
    plt.plot(recall, precision, label=f"AUPRC = {auprc:.4f}")
    plt.hlines(baseline, 0, 1, linestyle="--", label=f"Baseline = {baseline:.4f}")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix}Precision–Recall Curve".strip())
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return {"auroc": float(auroc), "auprc": float(auprc), "baseline": float(baseline)}

# Example:
# metrics = plot_roc_pr(all_labels, all_logits, title_prefix="Test | ")
# print(metrics)

In [ ]:
metrics = plot_roc_pr(all_labels, all_logits, title_prefix="Test | ")
print(metrics)